# SPARQL Service Health Check

**Project:** Climate Knowledge Graph  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)

**Purpose:** Verify that the SPARQL query service at `https://query.runstop.uk/` is reachable and returning results before running upload notebooks.

In [1]:
import os, requests
from dotenv import load_dotenv
from pathlib import Path

load_dotenv(Path('../../.env'))

SPARQL_URL = os.getenv('SPARQL_URL', 'https://query.runstop.uk/sparql')
print(f'SPARQL endpoint: {SPARQL_URL}')

SPARQL endpoint: https://query.runstop.uk/sparql


## Step 1 — Ping the endpoint

Send a minimal `ASK` query. Expects HTTP 200 and `{ "boolean": true }`.

In [2]:
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': 'ASK { ?s ?p ?o }', 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=15,
    )
    resp.raise_for_status()
    data = resp.json()
    if data.get('boolean') is True:
        print('✓ SPARQL service is UP — ASK query returned true')
    else:
        print(f'⚠ Unexpected response: {data}')
except requests.exceptions.ConnectionError as e:
    print(f'✗ Connection error: {e}')
except requests.exceptions.Timeout:
    print('✗ Request timed out after 15 s')
except Exception as e:
    print(f'✗ Error: {e}')

✓ SPARQL service is UP — ASK query returned true


## Step 2 — Count items in Wikibase

Returns the total number of items (`wd:Q*`) currently in the triplestore.

In [3]:
count_query = '''
SELECT (COUNT(?item) AS ?count)
WHERE {
  ?item ?p ?o .
}
'''
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': count_query, 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=30,
    )
    resp.raise_for_status()
    bindings = resp.json()['results']['bindings']
    count = bindings[0]['count']['value'] if bindings else 'unknown'
    print(f'Items in Wikibase: {count}')
except Exception as e:
    print(f'✗ Error: {e}')


Items in Wikibase: 14268


## Step 3 — Browse items with labels

Fetches up to 10 items that have an `rdfs:label`, showing QID, label (with language tag), and description — giving a human-readable snapshot of what is in the triplestore. If no labelled items are found, falls back to listing sample IRIs to confirm the triplestore is not completely empty.

In [4]:
browse_query = '''
SELECT ?item ?label ?description WHERE {
  ?item rdfs:label ?label .
  OPTIONAL { ?item schema:description ?description }
}
LIMIT 10
'''
try:
    resp = requests.get(
        SPARQL_URL,
        params={'query': browse_query, 'format': 'json'},
        headers={'Accept': 'application/sparql-results+json'},
        timeout=30,
    )
    resp.raise_for_status()
    bindings = resp.json()['results']['bindings']
    if bindings:
        print(f'✓ Found {len(bindings)} item(s) with labels:\n')
        col_w = (12, 35, 60)
        header = f"{'QID':<{col_w[0]}} {'Label':<{col_w[1]}} {'Description':<{col_w[2]}}"
        print(header)
        print('-' * sum(col_w))
        for b in bindings:
            qid   = b['item']['value'].split('/')[-1]
            label = b['label']['value']
            lang  = b['label'].get('xml:lang', '')
            desc  = b.get('description', {}).get('value', '—')
            label_display = f'{label} [{lang}]' if lang else label
            print(f'{qid:<{col_w[0]}} {label_display:<{col_w[1]}} {desc:<{col_w[2]}}')
    else:
        # Fall back: show a sample of raw subjects to confirm data exists
        print('⚠ No labelled items found. Checking for any triples...')
        fallback = '''SELECT DISTINCT ?s WHERE { ?s ?p ?o . FILTER(isIRI(?s)) } LIMIT 5'''
        r2 = requests.get(SPARQL_URL, params={'query': fallback, 'format': 'json'},
                          headers={'Accept': 'application/sparql-results+json'}, timeout=15)
        r2.raise_for_status()
        fb_bindings = r2.json()['results']['bindings']
        if fb_bindings:
            print('   Sample IRIs present in the triplestore:')
            for b in fb_bindings:
                print(f'   {b["s"]["value"]}')
        else:
            print('   Triplestore appears to be empty — no data uploaded yet.')
except Exception as e:
    print(f'✗ Error: {e}')

⚠ No labelled items found. Checking for any triples...
   Sample IRIs present in the triplestore:
   https://wikibase.runstop.uk
